# Credit Card Fraud Detection - Model Selection

This notebook compares three classification models:

- K-Nearest Neighbors (KNN)
- Random Forest
- Support Vector Machine (SVM)

The models will be compared using Accuracy, Precision, Recall, F1 Score, and ROC-AUC.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay
)

## Step 1: Load the Cleaned Dataset

Load the cleaned credit card fraud dataset produced during preprocessing.

In [ ]:
df = pd.read_csv("Data/credit_fraud_data.csv")

print(df.shape)
df.head()

## Step 2: Check Dataset Columns

Check the available columns and identify the fraud target column.

In [ ]:
df.columns.tolist()

## Step 3: Separate Features and Target

Separate the target variable from the input features.

`X` contains the features used for prediction and `y` contains the fraud labels.

In [ ]:
target_col = "is_fraud"

In [ ]:
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass counts:")
print(y.value_counts())

print("\nClass percentages:")
print(y.value_counts(normalize=True))

## Step 4: Remove Datetime Columns

Some columns contain dates and times stored as text.

These raw datetime columns are removed because the machine learning models require numeric or encoded features.

In [ ]:
datetime_columns = []

for col in X.select_dtypes(include="object").columns:
    converted = pd.to_datetime(X[col], errors="coerce")

    if converted.notna().mean() > 0.8:
        datetime_columns.append(col)

print("Datetime columns removed:", datetime_columns)

X = X.drop(columns=datetime_columns)

In [ ]:
print("Remaining non-numeric columns:")
print(X.select_dtypes(exclude=np.number).columns.tolist())

## Step 5: Prepare Numerical and Categorical Features

Numerical features are standardized.

Categorical features are converted into numeric values using One-Hot Encoding so they can be used by the classification models.

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(exclude=np.number).columns

print("Number of numerical features:", len(numeric_features))
print("Categorical features:", categorical_features.tolist())

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

## Step 6: Split the Dataset

Split the data into training and testing sets.

Stratification keeps approximately the same fraud and non-fraud proportions in both datasets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## Step 7: Create the Classification Models

Create the three target models:

1. K-Nearest Neighbors
2. Random Forest
3. Support Vector Machine

The preprocessing step is included in each model pipeline.

In [ ]:
from sklearn.svm import LinearSVC

models = {
    "KNN": make_pipeline(
        preprocessor,
        KNeighborsClassifier(
            n_neighbors=5,
            n_jobs=-1
        )
    ),

    "Random Forest": make_pipeline(
        preprocessor,
        RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        )
    ),

    "SVM": make_pipeline(
        preprocessor,
        LinearSVC(
            class_weight="balanced",
            random_state=42,
            max_iter=5000,
            dual="auto"
        )
    )
}

## Step 8: Set Up Cross-Validation

Use 5-fold Stratified Cross-Validation to evaluate each model.

The models are compared using:

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

## Step 9: Compare the Models

Evaluate KNN, Random Forest, and SVM using cross-validation.

The average scores are stored in a table for comparison.

In [ ]:
results = []

for name, model in models.items():

    print("Running:", name)

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })

    print("Finished:", name)

results_df = pd.DataFrame(results)

results_df

## Step 10: Visualize Model Performance

Create a bar chart to compare the evaluation scores of the three models.

In [ ]:
results_df.set_index("Model")[
    ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
].plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("KNN vs Random Forest vs SVM")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()

plt.show()

## Step 11: Select the Best Model

Select the model with the highest F1 Score.

F1 Score considers both Precision and Recall, making it useful for fraud detection.

In [ ]:
results_df = results_df.sort_values(
    by="F1",
    ascending=False
)

results_df

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Best model:", best_model_name)

## Step 12: Train the Best Model

Train the selected model using the complete training dataset and make predictions on the test dataset.

In [ ]:
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

## Step 13: Final Model Evaluation

Evaluate the selected model using Accuracy, Precision, Recall, F1 Score, and a Classification Report.

Recall is especially important because it measures how many actual fraud cases were detected.

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Step 14: ROC-AUC

Calculate the ROC-AUC score of the final model.

ROC-AUC measures how well the model distinguishes fraud from non-fraud transactions.

In [ ]:
if hasattr(best_model, "predict_proba"):

    y_prob = best_model.predict_proba(X_test)[:, 1]

    print(
        "ROC-AUC:",
        roc_auc_score(y_test, y_prob)
    )

## Step 15: Confusion Matrix

Display the confusion matrix for the selected model.

False negatives are especially important because they represent fraudulent transactions that the model failed to detect.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred
)

plt.title(best_model_name + " Confusion Matrix")

plt.show()

## Conclusion

KNN, Random Forest, and SVM were compared using Accuracy, Precision, Recall, F1 Score, and ROC-AUC.

The best model was selected using F1 Score because fraud detection requires a balance between detecting fraudulent transactions and avoiding incorrect fraud predictions.

Recall is also important because it shows how many actual fraudulent transactions were successfully detected.